# Importing All Dependencies

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Data Collection

In [2]:
raw_data=pd.read_csv('/content/mail_data.csv')

# Data Pre-processing

In [3]:
raw_data.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
raw_data.shape

(5572, 2)

In [5]:
raw_data.isnull().sum()

,0
Category,0
Message,0


### Data Transformation
Convert 'Category' column to numerical labels.

In [6]:
raw_data['Category'] = raw_data['Category'].map({'ham': 0, 'spam': 1})
display(raw_data.head())

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


### Text Vectorization
Convert the 'Message' column into numerical features using `TfidfVectorizer`.

In [7]:
X = raw_data['Message']
Y = raw_data['Category']

# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)

# Fit and transform the 'Message' column
X_vectorized = vectorizer.fit_transform(X)

print('Shape of X_vectorized:', X_vectorized.shape)
print('Shape of Y:', Y.shape)

Shape of X_vectorized: (5572, 8440)
Shape of Y: (5572,)


### Split Data
Divide the dataset into training and testing sets.

In [8]:
X_train, X_test, Y_train, Y_test = train_test_split(X_vectorized, Y, test_size=0.2, random_state=42)

print('Shape of X_train:', X_train.shape)
print('Shape of X_test:', X_test.shape)
print('Shape of Y_train:', Y_train.shape)
print('Shape of Y_test:', Y_test.shape)

Shape of X_train: (4457, 8440)
Shape of X_test: (1115, 8440)
Shape of Y_train: (4457,)
Shape of Y_test: (1115,)


### Model Selection and Training
Choose a suitable classification model and train it on the training data.

In [9]:
# Initialize the Multinomial Naive Bayes model
model = MultinomialNB()

# Train the model
model.fit(X_train, Y_train)

print("Multinomial Naive Bayes model trained successfully.")

Multinomial Naive Bayes model trained successfully.


### Model Evaluation
Assess the trained model's performance on the test set.

In [10]:
# Make predictions on the test set
Y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(Y_test, Y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Display classification report for more detailed metrics
print("\nClassification Report:")
print(classification_report(Y_test, Y_pred))

Model Accuracy: 97.76%

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       966
           1       1.00      0.83      0.91       149

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



### Prediction
Now, let's use the trained model to predict whether a new message is 'ham' or 'spam'.

In [11]:
# Function to predict if a message is spam or ham
def predict_message(message):
    # Vectorize the input message using the fitted TfidfVectorizer
    message_vectorized = vectorizer.transform([message])

    # Make prediction using the trained model
    prediction = model.predict(message_vectorized)

    # Return the category based on the numerical prediction
    if prediction[0] == 0:
        return 'Ham'
    else:
        return 'Spam'

# Test with a new message
new_message1 = "Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
print(f"Message: '{new_message1}'\nPrediction: {predict_message(new_message1)}\n")

new_message2 = "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"
print(f"Message: '{new_message2}'\nPrediction: {predict_message(new_message2)}")

Message: 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'
Prediction: Ham

Message: 'Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's'
Prediction: Spam


In [13]:
import pickle

# Save the trained model to a file in .pkl format
model_filename_pkl = 'spam_ham_model.pkl'
with open(model_filename_pkl, 'wb') as file:
    pickle.dump(model, file)

print(f"Model saved successfully to {model_filename_pkl}")

Model saved successfully to spam_ham_model.pkl
